# Task 8

Классификация `cats`/`dogs` с использованием предобученной CNN-модели.

Что делает ноутбук:
1. Скачивает архивы `train.zip`, `test1.zip` (Яндекс.Диск) и архив весов модели.
2. Распаковывает их в папке `task-8`.
3. Автоматически находит директорию `train` с файлами `cat/dog.10700..10799`.
4. Загружает веса в архитектуру сети и считает метрики.
5. Выводит `accuracy`, `f1_macro`, `P(cat.10727 -> cats)`, `P(dog.10727 -> dogs)`.

In [6]:
import json
import zipfile
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import urlopen, urlretrieve

import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Conv2D,
    MaxPooling2D,
    Activation,
    Dropout,
    Flatten,
    Dense,
)
from tensorflow.keras import backend as K

# Пути внутри task-8
BASE_DIR = Path.cwd()
PROJECT_ROOT = BASE_DIR.parent

WEIGHTS_URL = "https://storage.yandexcloud.net/lms-itmo-ru-files-27a87tyf/Computer_Vision/Task_2/Model_10k_images_cats_dogs_trained.zip"
TRAIN_PUBLIC_URL = "https://disk.yandex.ru/d/yBVJpUntzmJbfw"
TEST_PUBLIC_URL = "https://disk.yandex.ru/d/ML2MqS9yx-xeRg"

ARCHIVE_PATH = BASE_DIR / "Model_10k_images_cats_dogs_trained.zip"
EXTRACT_DIR = BASE_DIR / "Model_10k_images_cats_dogs_trained"
TRAIN_ARCHIVE_PATH = BASE_DIR / "train.zip"
TEST_ARCHIVE_PATH = BASE_DIR / "test1.zip"
TRAIN_EXTRACT_DIR = BASE_DIR / "train_data"
TEST_EXTRACT_DIR = BASE_DIR / "test_data"

IMG_WIDTH, IMG_HEIGHT = 150, 150

In [7]:
def _yandex_public_download_url(public_url: str) -> str:
    api_url = (
        "https://cloud-api.yandex.net/v1/disk/public/resources/download?"
        + urlencode({"public_key": public_url})
    )
    with urlopen(api_url) as response:
        data = json.loads(response.read().decode("utf-8"))

    href = data.get("href")
    if not href:
        raise RuntimeError(f"Не удалось получить ссылку для скачивания: {public_url}")
    return href


def _download_public_file(public_url: str, target_path: Path) -> None:
    if target_path.exists():
        return

    print(f"Downloading: {target_path.name}")
    href = _yandex_public_download_url(public_url)
    urlretrieve(href, target_path)


def _extract_zip(archive_path: Path, target_dir: Path) -> None:
    target_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path, "r") as zf:
        zf.extractall(target_dir)


def ensure_weights() -> Path:
    """Скачивает и распаковывает архив с весами при необходимости."""
    if not ARCHIVE_PATH.exists():
        print(f"Downloading: {WEIGHTS_URL}")
        urlretrieve(WEIGHTS_URL, ARCHIVE_PATH)

    _extract_zip(ARCHIVE_PATH, EXTRACT_DIR)

    weight_candidates = sorted(EXTRACT_DIR.rglob("*.h5"))
    if not weight_candidates:
        raise FileNotFoundError("В распакованном архиве не найден файл весов *.h5")

    # В архиве обычно лежит один файл весов, берем его.
    return weight_candidates[0]


def ensure_datasets() -> None:
    """Скачивает и распаковывает train/test архивы (если их еще нет)."""
    _download_public_file(TRAIN_PUBLIC_URL, TRAIN_ARCHIVE_PATH)
    _extract_zip(TRAIN_ARCHIVE_PATH, TRAIN_EXTRACT_DIR)

    _download_public_file(TEST_PUBLIC_URL, TEST_ARCHIVE_PATH)
    _extract_zip(TEST_ARCHIVE_PATH, TEST_EXTRACT_DIR)


def _has_required_train_images(path: Path) -> bool:
    return (path / "cat.10727.jpg").exists() and (path / "dog.10727.jpg").exists()


def resolve_train_dir() -> Path:
    """Ищет папку train, в которой есть нужные изображения cat/dog.10727.jpg."""
    direct_candidates = [
        BASE_DIR / "train",
        PROJECT_ROOT / "train",
        PROJECT_ROOT / "task-8" / "train",
        TRAIN_EXTRACT_DIR / "train",
        TRAIN_EXTRACT_DIR,
    ]

    for candidate in direct_candidates:
        if candidate.exists() and _has_required_train_images(candidate):
            return candidate

    for candidate in TRAIN_EXTRACT_DIR.rglob("*"):
        if candidate.is_dir() and _has_required_train_images(candidate):
            return candidate

    searched = "\n".join(str(path) for path in direct_candidates)
    raise FileNotFoundError(
        "Не найдена папка train с файлами cat.10727.jpg и dog.10727.jpg.\n"
        "Проверьте, что train.zip успешно скачан/распакован. Проверенные пути:\n"
        f"{searched}"
    )


def build_model(input_shape):
    model = Sequential([
        Input(shape=input_shape),
        Conv2D(32, (3, 3)),
        Activation("relu"),
        MaxPooling2D(pool_size=(2, 2)),

        Conv2D(32, (3, 3)),
        Activation("relu"),
        MaxPooling2D(pool_size=(2, 2)),

        Conv2D(64, (3, 3)),
        Activation("relu"),
        MaxPooling2D(pool_size=(2, 2)),

        Flatten(),
        Dense(64),
        Activation("relu"),
        Dropout(0.5),
        Dense(1),
        Activation("sigmoid"),
    ])
    model.compile(loss="binary_crossentropy", optimizer="rmsprop", metrics=["accuracy"])
    return model

In [8]:
if K.image_data_format() == "channels_first":
    input_shape = (3, IMG_WIDTH, IMG_HEIGHT)
else:
    input_shape = (IMG_WIDTH, IMG_HEIGHT, 3)

ensure_datasets()
weights_path = ensure_weights()
train_dir = resolve_train_dir()
print(f"Using train dir: {train_dir}")

model = build_model(input_shape)
model.load_weights(weights_path)

X, y, filenames = [], [], []
for i in range(10700, 10800):
    cat_name = f"cat.{i}.jpg"
    dog_name = f"dog.{i}.jpg"

    for name, label in ((cat_name, 0), (dog_name, 1)):
        image_path = train_dir / name
        if not image_path.exists():
            raise FileNotFoundError(f"Не найден файл: {image_path}")

        image = load_img(image_path, target_size=(IMG_WIDTH, IMG_HEIGHT))
        arr = img_to_array(image) / 255.0
        X.append(arr)
        y.append(label)
        filenames.append(name)

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int32)

pred = model.predict(X, verbose=0).ravel()
y_pred = (pred >= 0.5).astype(int)

acc = accuracy_score(y, y_pred)
f1 = f1_score(y, y_pred, average="macro")

cat_idx = filenames.index("cat.10727.jpg")
dog_idx = filenames.index("dog.10727.jpg")

p_cat_is_dog = float(pred[cat_idx])
p_dog_is_dog = float(pred[dog_idx])

p_cat_is_cat = 1.0 - p_cat_is_dog

print("accuracy =", round(acc, 3))
print("f1_macro =", round(f1, 3))
print("P(cat.10727 -> cats) =", round(p_cat_is_cat, 3))
print("P(dog.10727 -> dogs) =", round(p_dog_is_dog, 3))

Downloading: train.zip
Downloading: test1.zip
Using train dir: /Users/afafos/Desktop/Program/itmo/image-processing-cv-itmo/task-8/train_data/train
accuracy = 0.76
f1_macro = 0.755
P(cat.10727 -> cats) = 0.928
P(dog.10727 -> dogs) = 0.978
